# FISH bulk processing
From microscope images to mRNA counts per cell (and per mother and bud):

1. **Read** the .vsi and DIC images (several images at the same time)
2. **Segment** cells and nuclei (Omnipose, on the GPU if available)
3. **Detect spots**, align the masks, assign buds to mothers (bud model) and count spots per cell
4. **Check and correct** in napari
5. **Reassign spots** to the corrected masks

Each step writes its results to the image's folder in `output/` and records what it did in `img.json`, so you can
stop after any step and continue later.

The work for one image is in short functions in `buddy/pipeline.py` (one per step). The napari
**buddy pipeline** panel (Plugins → napari-buddy) runs the same functions, so both give the same result.

In [1]:
import time, logging
from pathlib import Path
from buddy.experiment import Experiment
from buddy.image import Image
from buddy import pipeline                    # the processing steps for one image
from buddy.rerun_check import select_images   # skips images already processed with the same settings
from buddy.parallel import default_workers, run_parallel, read_one, spots_one
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")
WORKERS = default_workers(gb_per_image=4)     # images read at the same time (limited by CPU cores and free memory)
print('parallel workers:', WORKERS)

# step 3: None = automatic (one image at a time on the GPU when GPU spot detection is on, otherwise in parallel);
# True / False = always parallel / always one at a time
PARALLEL_SPOTS = None

parallel workers: 4


***

### 1. Read in microscope images

Each step skips images that were already processed with the same settings. Images processed with other
settings are listed with what changed, and you are asked whether to overwrite them (config: `"if_already_processed": "ask" | "skip" | "overwrite"`).


In [2]:
CFG_FILE = r"..\buddy\config_files\220601\config220601-yET563-BR1.json"
exp = Experiment.from_cfg_file(CFG_FILE) #### important file contains threshold values! 
######################################################################################## and other fun parameters!!!!!!!!!!!!!!!!!
exp.inputdir = r"..\..\test yourself\small example set\YET572 BR4"
exp.outputdir = r"..\..\test yourself\small example set\YET572 BR4\output"
exp.create_image_list()

10/11/2026 03:47:44PM [INFO] found cfg file config_files\220601\config220601-yET563-BR1.json
10/11/2026 03:47:44PM [INFO] getting .vsi files from ..\..\test yourself\small example set\YET572 BR4
10/11/2026 03:47:44PM [INFO] examining image ..\..\test yourself\small example set\YET572 BR4\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03_CY5, CY3.5 NAR, CY3, DAPI.vsi
10/11/2026 03:47:44PM [INFO] ..found VSI directory _YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03_CY5, CY3.5 NAR, CY3, DAPI_
10/11/2026 03:47:44PM [INFO] ..found DIC file YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03_DIC.tif
10/11/2026 03:47:44PM [INFO] ..all good, adding image to image list
10/11/2026 03:47:44PM [INFO] examining image ..\..\test yourself\small example set\YET572 BR4\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02_CY5, CY3.5 NAR, CY3, DAPI.vsi
10/11/2026 03:47:44PM [INFO] ..found VSI directory _YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02_CY5, CY3.5 NAR, CY3, DAPI_
10/11/2026 03:47:44PM [INFO] ..found DIC f

In [ ]:
mode = f'parallel ({WORKERS} workers)'
todo = select_images(exp, 'read')       # new images + (if you agree) images processed with other settings
jobs = [(params, CFG_FILE, exp.inputdir, exp.outputdir, mode) for params in todo]
run_parallel(read_one, jobs, WORKERS)

***

### 2. Segment images (GPU, one image after the other)
Per image (`pipeline.segment_image`): cell masks from a DIC + DAPI composite and nucleus masks from DAPI
(Omnipose), then small objects are removed and, if set, cells touching the image edge (`DIC_masks_pp.tif`).

In [ ]:
exp.read_image_list_from_jsons()
todo = select_images(exp, 'segment')
if todo:
    exp.init_omnipose()        # runs on the GPU when "use_gpu": true in the config and a GPU is available

for n, f in enumerate(todo, start=1):
    pipeline.segment_image(f, exp)     # segment_cells -> segment_dapi -> postprocess_masks
    print(f'  {n}/{len(todo)} done: {Path(f).parent.name}', flush=True)

***

### 3. Detect spots, then assign buds + count spots
**3a** (`pipeline.spots_image`), for all images: in-focus planes, background filter, spot detection,
decomposition of dense regions, and alignment of the cell masks to the spots (on the GPU one image at a time, or
in parallel without a GPU - see `PARALLEL_SPOTS`).

**3b** (`pipeline.buds_and_counts`), per image: the bud model makes the bud masks (if enabled in the config),
then the spots are counted per cell, mother and bud (`{channel}.csv`).

In [ ]:
from buddy.experiment import gpu_available
exp.read_image_list_from_jsons()
todo = select_images(exp, 'spots')
gpu_spots = exp.parameters.get('gpu_spot_detection', True) and gpu_available(exp.use_gpu)
parallel_spots = (not gpu_spots) if PARALLEL_SPOTS is None else PARALLEL_SPOTS
mode = (f'parallel ({WORKERS} workers)' if parallel_spots else 'serial') + (' + GPU spot detection' if gpu_spots else '')
print('step 3:', mode)

# 3a: find_focus -> filter -> detect_spots -> decompose_spots -> align
jobs = [(f, CFG_FILE, exp.inputdir, exp.outputdir, mode) for f in todo]
if parallel_spots:
    run_parallel(spots_one, jobs, WORKERS)
else:
    for n, f in enumerate(todo, start=1):
        pipeline.spots_image(f, exp, mode)
        print(f'  {n}/{len(todo)} done: {Path(f).parent.name}', flush=True)

# 3b: classify_buds (bud model) -> assign_spots
exp.init_bud_model()       # loads the bud model once (only if bud classification is enabled in the config)
for n, f in enumerate(todo, start=1):
    pipeline.buds_and_counts(f, exp)
    print(f'  {n}/{len(todo)} buds + counts done: {Path(f).parent.name}', flush=True)

***

### 4. Check and correct
Run the cell below: napari opens the first image with a **Correct images** panel
(**◀ previous** · **save** · **save + next ▶** · **skip ▶**).

For each image, check the boxed cells (`check these buds`; hover a box for the reason) and the `bud → mother links`,
and fix what is wrong: paint in `bud masks` (1 = bud, 3 = mother), or draw a line from a bud to its mother in the
**Reassign buds to mothers** panel. Then press **save + next**.
The **Layer groups** panel shows / hides groups of layers.


In [3]:
from napari_buddy import review_images

review_images(exp, start=0)     # start = image number to begin with (0 = first)

10/11/2026 03:48:00PM [INFO] created output dir ..\..\test yourself\small example set\YET572 BR4\output\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03
image 1/2: YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03
  NO BUD MODEL OUTPUT (step 3 has not run for this image): 4 small cell(s) pre-marked as bud by the old size rule (yellow, not a prediction), 0 non-round cell(s) circled (cyan)
  opened in 5.4 s (files 0.0 s, napari 2.1 s, other 3.2 s)
10/11/2026 03:50:26PM [INFO] found cfg file F:\buddy-code-for-submission\test yourself\config.json
10/11/2026 03:50:26PM [INFO] reading json file F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02\img.json
10/11/2026 03:50:26PM [INFO] reading json file F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03\img.json
step 3 (spots): 2 new, 0 already processed with these settings (skipped), 0 processed with o

image 1/2: YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03
  bud model result loaded; cells to check: 2 possible missed bud; high ASH1/CLB2: 4 high ASH1/CLB2, no bud, 2 late-M pair?
  opened in 1.4 s (files 0.3 s, napari 0.5 s, other 0.7 s)
saved corrected masks for YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03
  16 bud(s) linked to a mother (0 newly linked), 0 without mother
  saved in 2.2 s
10/11/2026 04:09:46PM [INFO] created output dir F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02
image 2/2: YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_02
  bud model result loaded; cells to check: 3 weak pair; high ASH1/CLB2: 4 high ASH1/CLB2, no bud, 1 late-M pair?
  opened in 1.2 s (files 0.0 s - read in advance, napari 0.6 s, other 0.7 s)
10/11/2026 04:09:58PM [INFO] created output dir F:\buddy-code-for-submission\test yourself\small example set\YET572 BR4\output\YET572_BR1_ASH1CLB2Q670_SRL1cal610_SUN4Q570_03
image 1/

***

### 5. Reassign spots
Count the spots again with the corrected masks from step 4 (`pipeline.reassign_image`).

In [ ]:
exp.read_image_list_from_jsons()
for n, f in enumerate(exp.json_files, start=1):
    pipeline.reassign_image(f, exp)      # reads corrected/ masks, rewrites {channel}.csv
    print(f'  {n}/{len(exp.json_files)} done: {Path(f).parent.name}', flush=True)